# Activity 7: LOESS and Uncertainty
Scatter two Google Trends searches, fit a LOESS line, and add a 95% confidence band.

In [ ]:
import numpy as np, pandas as pd, altair as alt, io, warnings
from statsmodels.nonparametric.smoothers_lowess import lowess
warnings.filterwarnings('ignore', category=RuntimeWarning)

# sample data to practice on
rng = np.random.default_rng(7); n = 120
termA = np.clip(rng.normal(50, 22, n), 2, 100)
termB = np.clip(12 + 0.9*termA - 0.004*(termA-40)**2 + rng.normal(0, 4 + 0.18*termA), 0, 100)
df = pd.DataFrame({'termA': termA, 'termB': termB})

# LOESS line with a bootstrapped 95% band
def loess_with_ci(x, y, span=0.5, n_boot=200, grid=60, seed=0):
    x = np.asarray(x, float); y = np.asarray(y, float)
    gx = np.linspace(x.min(), x.max(), grid)
    rng = np.random.default_rng(seed); fits = np.empty((n_boot, grid))
    for b in range(n_boot):
        i = rng.integers(0, len(x), len(x))
        sm = lowess(y[i], x[i], frac=span, return_sorted=True)
        fits[b] = np.interp(gx, sm[:, 0], sm[:, 1])
    base = lowess(y, x, frac=span, return_sorted=True)
    return pd.DataFrame({'x': gx, 'fit': np.interp(gx, base[:, 0], base[:, 1]),
        'lower': np.nanpercentile(fits, 2.5, axis=0), 'upper': np.nanpercentile(fits, 97.5, axis=0)})

# Google Trends CSV loader
def load_trends(path):
    lines = open(path, encoding='utf-8-sig').read().splitlines()
    start = next(i for i, l in enumerate(lines) if l.count(',') >= 2)
    g = pd.read_csv(io.StringIO('\n'.join(lines[start:])))
    names = list(g.columns[1:3])
    g.columns = ['date', 'termA', 'termB'] + list(g.columns[3:])
    for c in ['termA', 'termB']:
        g[c] = pd.to_numeric(g[c].astype(str).str.replace('<1', '0.5'), errors='coerce')
    return g[['date', 'termA', 'termB']].dropna(), names

## 1. Scatter plot
Each point is one time period: term A vs. term B.

In [ ]:
alt.Chart(df).mark_circle(opacity=0.5).encode(
    x=alt.X('termA:Q', title='Term A'), y=alt.Y('termB:Q', title='Term B')
).properties(width=440, height=320)

alt.Chart(...)

## 2. Add the LOESS line and 95% band

In [ ]:
band = loess_with_ci(df['termA'], df['termB'], span=0.5)

pts  = alt.Chart(df).mark_circle(opacity=0.4).encode(x='termA:Q', y='termB:Q')
ci   = alt.Chart(band).mark_area(opacity=0.3, color='#c66').encode(x='x:Q', y='lower:Q', y2='upper:Q')
line = alt.Chart(band).mark_line(color='firebrick').encode(x='x:Q', y='fit:Q')
(ci + line + pts).properties(width=440, height=320)

alt.LayerChart(...)

## 3. Read the band
Where is the band narrow? Where is it wide? Find a range of x values where you would not trust the line.

## 4. Change the span
Rerun step 2 with `span=0.2`, then `span=0.8`. Compare the line and the band each time.

In [ ]:
# try different span values

band = loess_with_ci(df['termA'], df['termB'], span=0.2)

pts  = alt.Chart(df).mark_circle(opacity=0.4).encode(x='termA:Q', y='termB:Q')
ci   = alt.Chart(band).mark_area(opacity=0.3, color='#c66').encode(x='x:Q', y='lower:Q', y2='upper:Q')
line = alt.Chart(band).mark_line(color='firebrick').encode(x='x:Q', y='fit:Q')
(ci + line + pts).properties(width=440, height=320)

alt.LayerChart(...)

## 5. Use your own Google Trends data
1. Go to [trends.google.com](https://trends.google.com) and enter two search terms you think might be related.
2. Choose a country and time range.
3. Download the CSV from the *Interest over time* chart.
4. Upload it to Colab using the folder icon on the left.
5. Put your filename below and run the cell.

In [ ]:
df, names = load_trends('/content/time_series_US_20210924-2021_20260924-2021.csv')
print(names, len(df), 'rows')

band = loess_with_ci(df['termA'], df['termB'], span=0.5)
pts  = alt.Chart(df).mark_circle(opacity=0.4).encode(
    x=alt.X('termA:Q', title=names[0]), y=alt.Y('termB:Q', title=names[1]),
    tooltip=['date:N', 'termA:Q', 'termB:Q'])
ci   = alt.Chart(band).mark_area(opacity=0.3, color='#c66').encode(x='x:Q', y='lower:Q', y2='upper:Q')
line = alt.Chart(band).mark_line(color='firebrick').encode(x='x:Q', y='fit:Q')
(ci + line + pts).properties(width=440, height=320, title=f'{names[0]} vs. {names[1]}')

['Costumes', 'Horror'] 61 rows


alt.LayerChart(...)

## 6. Write it up
In 2 to 3 sentences: are the two terms related, and where does the band say you can or can't trust that?

## Submit
Post your chart and your 2 to 3 sentences in **#07-loess**. Include your two search terms and how you filtered the data (country, time range, search type).